In [ ]:
import pandas as pd
import re
import numpy as np
from collections import Counter

# ==== Load ====
path = "/Users/aniluchavez/Documents/Language/Python/Final_words_english_only_BERT/PTYFK_task40_words_english_only/PTYFK_task40_filtered_used_rows_withNP_withClusterIDNew.xlsx"
df = pd.read_excel(path)

# ==== Identify speaker columns ====
speaker_cols = [c for c in df.columns if re.fullmatch(r"Speaker\d+", str(c))]
speaker_cols = sorted(speaker_cols, key=lambda x: int(x.replace("Speaker", "")))

speaker1_col = "Speaker1"
other_cols = [c for c in speaker_cols if c != speaker1_col]

# ==== Regex tokenization ====
# Keeps words + contractions like can't, it's
token_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def cell_to_tokens(x):
    if pd.isna(x):
        return []
    return token_re.findall(str(x).strip().lower())

def row_speaker_and_tokens(row):
    """
    Each row has the word in exactly one SpeakerX column.
    Find which speaker column is filled and tokenize via regex.
    """
    for col in speaker_cols:
        val = row.get(col, np.nan)
        if pd.notna(val) and str(val).strip() != "":
            return col, cell_to_tokens(val)
    return None, []

# ==== Build per-speaker token lists ====
per_speaker_tokens = {c: [] for c in speaker_cols}
for _, row in df.iterrows():
    spk, toks = row_speaker_and_tokens(row)
    if spk is not None:
        per_speaker_tokens[spk].extend(toks)

# ==== Sets: Speaker1 vs Others ====
spk1_set = set(per_speaker_tokens[speaker1_col])
others_set = set().union(*[set(per_speaker_tokens[c]) for c in other_cols])

shared = spk1_set & others_set

print(f"unique Speaker1 words: {len(spk1_set)}")
print(f"unique Others words:   {len(others_set)}")
print(f"shared words:          {len(shared)}")

# ==== Counts (helpful for prioritizing) ====
spk1_counts = Counter(per_speaker_tokens[speaker1_col])
others_counts = Counter()
for c in other_cols:
    others_counts.update(per_speaker_tokens[c])

shared_df = (
    pd.DataFrame([{
        "word": w,
        "count_speaker1": spk1_counts[w],
        "count_others_total": others_counts[w],
        "count_total": spk1_counts[w] + others_counts[w],
    } for w in sorted(shared)])
    .sort_values(["count_total", "count_speaker1", "count_others_total"], ascending=False)
)

# ==== Annotate original rows with shared flag ====
def row_primary_word(row):
    for col in speaker_cols:
        val = row.get(col, np.nan)
        if pd.notna(val) and str(val).strip() != "":
            toks = cell_to_tokens(val)
            return toks[0] if toks else np.nan
    return np.nan

def row_speaker(row):
    for col in speaker_cols:
        val = row.get(col, np.nan)
        if pd.notna(val) and str(val).strip() != "":
            return col
    return np.nan

df_out = df.copy()
df_out["row_speaker"] = df_out.apply(row_speaker, axis=1)
df_out["row_word_regex"] = df_out.apply(row_primary_word, axis=1)
df_out["shared_spk1_vs_others"] = df_out["row_word_regex"].isin(shared)

# ==== Save ====
out_path = "shared_words_speaker1_vs_others.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    shared_df.to_excel(writer, sheet_name="shared_words", index=False)
    df_out.to_excel(writer, sheet_name="annotated_rows", index=False)

print("Saved:", out_path)

In [ ]:
import re
import numpy as np
import pandas as pd
import sys
from collections import Counter
sys.path.append("/Users/aniluchavez/Documents/Language/Python")

from spike_processing_utils import (
    load_mat_data,
    get_cells_by_region,
    extract_speaker_events,
    compute_spike_sums,
)

TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")  # keeps contractions like can't, it's

def _cell_to_first_token(x):
    if pd.isna(x):
        return np.nan
    toks = TOKEN_RE.findall(str(x).strip().lower())
    return toks[0] if toks else np.nan

def _get_speaker_cols(df):
    cols = [c for c in df.columns if re.fullmatch(r"Speaker\d+", str(c))]
    return sorted(cols, key=lambda x: int(x.replace("Speaker", "")))

def _words_in_extraction_order(df, speaker_col):
    """
    Matches extract_speaker_events ordering:
    rows where df[speaker_col] is non-empty, in original row order.
    """
    tmp = df.copy()
    tmp[speaker_col] = tmp[speaker_col].astype(str).str.strip()
    spk_df = tmp[tmp[speaker_col] != ""]
    return [ _cell_to_first_token(x) for x in spk_df[speaker_col].values ]

def _shared_word_set(df, speaker1_col="Speaker1", min_repeats_each_side=1):
    speaker_cols = _get_speaker_cols(df)
    other_cols = [c for c in speaker_cols if c != speaker1_col]

    spk1_words = [w for w in _words_in_extraction_order(df, speaker1_col) if pd.notna(w)]
    other_words = []
    for spk in other_cols:
        other_words.extend([w for w in _words_in_extraction_order(df, spk) if pd.notna(w)])

    c1 = Counter(spk1_words)
    co = Counter(other_words)

    shared = set(c1.keys()) & set(co.keys())
    shared = {w for w in shared if (c1[w] >= min_repeats_each_side and co[w] >= min_repeats_each_side)}

    return shared, speaker_cols

def make_shared_counts(words_self, words_other):
    c_self = Counter(words_self)
    c_other = Counter(words_other)
    shared = sorted(set(c_self.keys()) & set(c_other.keys()))
    counts_df = pd.DataFrame({
        "word": shared,
        "n_self": [c_self[w] for w in shared],
        "n_other": [c_other[w] for w in shared],
        "n_total": [c_self[w] + c_other[w] for w in shared],
    }).sort_values(["n_total", "word"], ascending=[False, True]).reset_index(drop=True)
    return shared, counts_df

def build_neuron_by_sharedword_matrices_fast(
    excel_path: str,
    mat_path: str,
    region_ranges: dict,
    region_name: str,
    speaker1_col: str = "Speaker1",
    window_spec_self: dict = None,
    window_spec_other: dict = None,
    average_repeats: bool = True,
    min_repeats_each_side: int = 1,
    bin_mode: str = "explicit_event_bounds",
):
    """
    If average_repeats=True:
        returns FR_self, FR_other, col_labels, neuron_ids, shared_counts_df
        where FR_* is [n_neurons x n_unique_shared_words]
        col_labels is the ACTUAL shared word list used as columns
        shared_counts_df has word -> n_self, n_other, n_total

    If average_repeats=False:
        returns FR_self, FR_other, (cols_self, cols_other), neuron_ids, shared_counts_df
        where FR_self is [n_neurons x n_events_self_shared]
              FR_other is [n_neurons x n_events_other_shared]
              cols_self/cols_other are occurrence-labeled columns (word__occK)
    """

    # -----------------------
    # Defaults
    # -----------------------
    if window_spec_self is None:
        window_spec_self = dict(
            speaker_of_interest="Speaker1",
            mode="target_vs_other_fixed_window_from_ref",
            target_ref_point="onset",
            target_shift=250,
            target_window_length=500,
            other_ref_point="offset",
            other_shift=-500,
            other_window_length=300,
        )
    if window_spec_other is None:
        window_spec_other = dict(
            speaker_of_interest="Speaker1",
            mode="target_vs_other_fixed_window_from_ref",
            target_ref_point="onset",
            target_shift=250,
            target_window_length=500,
            other_ref_point="offset",
            other_shift=-500,
            other_window_length=300,
        )

    # -----------------------
    # Load transcript + shared set (global)
    # -----------------------
    df = pd.read_excel(excel_path, keep_default_na=False)
    shared, speaker_cols = _shared_word_set(
        df,
        speaker1_col=speaker1_col,
        min_repeats_each_side=min_repeats_each_side,
    )
    other_cols = [c for c in speaker_cols if c != speaker1_col]

    # -----------------------
    # Load spikes + neurons
    # -----------------------
    spikes, qual, chan = load_mat_data(mat_path)
    region_cells_all = get_cells_by_region(chan, qual, region_ranges)
    if region_name not in region_cells_all:
        raise ValueError(f"Region '{region_name}' not found")

    region_cells = {region_name: region_cells_all[region_name]}
    n_neurons = len(region_cells_all[region_name])
    neuron_ids = list(range(n_neurons))

    # -----------------------
    # PRECOMPUTE EVENTS (key optimization)
    # -----------------------
    events_self_all = extract_speaker_events(excel_path, **window_spec_self)
    events_other_all = extract_speaker_events(excel_path, **window_spec_other)

    # -----------------------
    # Speaker1 (self)
    # -----------------------
    ev_self = events_self_all.get(speaker1_col)
    if ev_self is None or len(ev_self) == 0:
        raise ValueError("No Speaker1 events")

    words_self_full = _words_in_extraction_order(df, speaker1_col)[: ev_self.shape[0]]
    mask_self = np.array([(w in shared) for w in words_self_full], dtype=bool)
    ev_self = ev_self[mask_self]
    words_self = np.array(words_self_full, dtype=object)[mask_self]

    # -----------------------
    # Others (pooled)
    # -----------------------
    ev_other_list, words_other_list = [], []
    for spk in other_cols:
        ev = events_other_all.get(spk)
        if ev is None or len(ev) == 0:
            continue

        words_full = _words_in_extraction_order(df, spk)[: ev.shape[0]]
        mask = np.array([(w in shared) for w in words_full], dtype=bool)

        if np.any(mask):
            ev_other_list.append(ev[mask])
            words_other_list.append(np.array(words_full, dtype=object)[mask])

    if not ev_other_list:
        raise ValueError("No shared-word events for other speakers")

    ev_other = np.vstack(ev_other_list)
    words_other = np.concatenate(words_other_list)

    # -----------------------
    # Derive shared words ACTUALLY PRESENT after masking
    # -----------------------
    shared_words_present, shared_counts_df = make_shared_counts(
        words_self.tolist(),
        words_other.tolist(),
    )

    # -----------------------
    # SANITY CHECKS: alignment + shared filtering
    # -----------------------
    assert ev_self.shape[0] == len(words_self), "Mismatch: ev_self rows != words_self length"
    assert ev_other.shape[0] == len(words_other), "Mismatch: ev_other rows != words_other length"
    assert all([(w in shared) for w in words_self if pd.notna(w)]), "Non-shared word leaked into SELF"
    assert all([(w in shared) for w in words_other if pd.notna(w)]), "Non-shared word leaked into OTHER"

    win_ms_self = ev_self[:, 2] - ev_self[:, 1]
    win_ms_other = ev_other[:, 2] - ev_other[:, 1]
    assert np.all(win_ms_self > 0), "Non-positive window length in SELF events"
    assert np.all(win_ms_other > 0), "Non-positive window length in OTHER events"

    print("SELF window lengths (ms):", np.unique(win_ms_self)[:10], "n_unique:", len(np.unique(win_ms_self)))
    print("OTHER window lengths (ms):", np.unique(win_ms_other)[:10], "n_unique:", len(np.unique(win_ms_other)))
    print("n SELF shared events:", len(words_self), "unique shared words present:", len(set(words_self)))
    print("n OTHER shared events:", len(words_other), "unique shared words present:", len(set(words_other)))

    # -----------------------
    # Compute FR per event per neuron
    # -----------------------
    def compute_fr(ev):
        sums = compute_spike_sums(spikes, region_cells, ev, mode=bin_mode)
        mat = sums[region_name]  # [events x neurons]
        win_ms = ev[:, 2] - ev[:, 1]
        win_s = np.where(win_ms > 0, win_ms / 1000.0, np.nan)
        return mat / win_s[:, None]  # [events x neurons]

    fr_self_events = compute_fr(ev_self)     # [Eself x N]
    fr_other_events = compute_fr(ev_other)   # [Eother x N]

    # -----------------------
    # CASE 1: keep repeats (occurrence-level columns)
    # -----------------------
    if not average_repeats:
        def make_occ_cols(words):
            c = Counter()
            cols = []
            for w in words:
                c[w] += 1
                cols.append(f"{w}__occ{c[w]}")
            return cols

        cols_self = make_occ_cols(words_self.tolist())
        cols_other = make_occ_cols(words_other.tolist())

        FR_self = fr_self_events.T   # [N x Eself]
        FR_other = fr_other_events.T # [N x Eother]

        return FR_self, FR_other, (cols_self, cols_other), neuron_ids, shared_counts_df

    # -----------------------
    # CASE 2: average repeats (word-level columns)
    # -----------------------
    col_labels = shared_words_present  # <-- KEY FIX: actual shared words used as columns

    FR_self = np.full((n_neurons, len(col_labels)), np.nan)
    FR_other = np.full((n_neurons, len(col_labels)), np.nan)

    for j, w in enumerate(col_labels):
        ii = np.where(words_self == w)[0]
        jj = np.where(words_other == w)[0]
        if ii.size:
            FR_self[:, j] = np.nanmean(fr_self_events[ii, :], axis=0)
        if jj.size:
            FR_other[:, j] = np.nanmean(fr_other_events[jj, :], axis=0)

    return FR_self, FR_other, col_labels, neuron_ids, shared_counts_df

In [ ]:
region_ranges = { "hippocampus": [(1,16),(25,40)],}
        # "ACC": [(17,24),(41,48)]}
# window_self = dict(
#     speaker_of_interest="Speaker1",
#     mode="target_vs_other_custom_bounds",
#     target_start_ref="onset",
#     target_start_shift=-200,
#     target_end_ref="offset",
#     target_end_shift=-200,
#     other_start_ref="onset",    # these "other_*" are unused for Speaker1 rows,
#     other_start_shift=200,        # but must exist as args
#     other_end_ref="offset",
#     other_end_shift=200,
# )
window_self= dict(
    speaker_of_interest="Speaker1",
    mode="target_vs_other_fixed_window_from_ref",
    target_ref_point="onset",
    target_shift=-150,
    target_window_length=500,
    other_ref_point="onset",
    other_shift=200,
    other_window_length=500,
)


window_other = dict(
    speaker_of_interest="Speaker1",
    mode="target_vs_other_fixed_window_from_ref",
    target_ref_point="onset",
    target_shift=-150,
    target_window_length=500,
    other_ref_point="onset",
    other_shift=200,
    other_window_length=500,
)

FR_self, FR_other, col_labels, neuron_ids, shared_counts_df = (
    build_neuron_by_sharedword_matrices_fast(
        excel_path="/Users/aniluchavez/Documents/Language/YEU/147/20250423_PTYEU_task147_convoFinalwithPuncta_withPOS.xlsx",
        mat_path='/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YEU/ptYEU_task147_new_spikes.mat',
        region_ranges=region_ranges,
        region_name="hippocampus",
        average_repeats=True,          # <-- IMPORTANT
        window_spec_self=window_self,
        window_spec_other=window_other,
    )
)

print("FR_self shape:", FR_self.shape)   # [n_neurons x n_shared_words]
print("FR_other shape:", FR_other.shape)
print("n shared words:", len(col_labels))
print("first 20 words:", col_labels[:30])

# sanity
assert FR_self.shape[1] == FR_other.shape[1] == len(col_labels)
assert set(shared_counts_df["word"]) == set(col_labels)

In [ ]:
from sklearn.metrics.pairwise import cosine_distances
import numpy as np
import pandas as pd

def word_word_cosine_distance(FR, col_labels, fill="col_mean"):
    """
    FR: [neurons x words]
    returns: DataFrame [words x words] cosine distance
    """
    X = FR.astype(float).copy()

    # handle NaNs
    if np.isnan(X).any():
        if fill == "col_mean":
            col_means = np.nanmean(X, axis=0, keepdims=True)
            X = np.where(np.isfinite(X), X, col_means)
        elif fill == "zero":
            X = np.nan_to_num(X, nan=0.0)
        else:
            raise ValueError("fill must be 'col_mean' or 'zero'")

    # cosine distances between word vectors (each word = column in FR)
    D = cosine_distances(X.T)  # [words x words]
    return pd.DataFrame(D, index=col_labels, columns=col_labels)

D_self  = word_word_cosine_distance(FR_self, col_labels)
D_other = word_word_cosine_distance(FR_other, col_labels)

print(D_self.iloc[:5, :5])
print(D_other.iloc[:5, :5])

In [ ]:
from scipy.stats import pearsonr, spearmanr
import numpy as np

# vectorize upper triangle (excluding diagonal)
iu = np.triu_indices(len(col_labels), k=1)

v_self  = D_self.values[iu]
v_other = D_other.values[iu]

# remove NaNs if present
mask = np.isfinite(v_self) & np.isfinite(v_other)
v_self  = v_self[mask]
v_other = v_other[mask]

# Pearson: linear geometry similarity
r_p, p_p = pearsonr(v_self, v_other)

# Spearman: rank-order geometry similarity
r_s, p_s = spearmanr(v_self, v_other)

print(f"Pearson r = {r_p:.3f}, p = {p_p:.2e}")
print(f"Spearman ρ = {r_s:.3f}, p = {p_s:.2e}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica"]
mpl.rcParams["font.size"] = 12

# critical for Illustrator editable fonts
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["pdf.fonttype"] = 42
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    v_self,
    v_other,
    s=14,
    alpha=0.45,
    color="#9467BD",
    edgecolors="none"
)

ax.set_xlabel("speaking word–word distance")
ax.set_ylabel("listening word–word distance")

ax.set_title(
    f"geometry similarity\nspearman ρ={r_s:.2f}",
    pad=14
)

ax.axline((0, 0), slope=1, linestyle="--", color="black", linewidth=1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

ax.set_aspect("equal", adjustable="box")
ax.tick_params(width=1.2, length=5)

plt.tight_layout()

plt.savefig(
    "geometry_rsasimilarityYEU.eps",
    format="eps",
    bbox_inches="tight"
)

plt.show()

In [ ]:
from scipy.stats import spearmanr, pearsonr
import numpy as np

iu = np.triu_indices(len(col_labels), k=1)

v_self  = D_self.values[iu]
v_other = D_other.values[iu]

mask = np.isfinite(v_self) & np.isfinite(v_other)
v_self  = v_self[mask]
v_other = v_other[mask]

r_obs_p, _ = pearsonr(v_self, v_other)
r_obs_s, _ = spearmanr(v_self, v_other)

print(f"Observed Pearson r = {r_obs_p:.3f}")
print(f"Observed Spearman ρ = {r_obs_s:.3f}")

In [ ]:
def geometry_permutation_test(
    D_self: pd.DataFrame,
    D_other: pd.DataFrame,
    n_perm: int = 10000,
    method: str = "spearman",
    seed: int = 0,
):
    """
    Permute word labels in OTHER and recompute geometry correlation.
    """
    rng = np.random.default_rng(seed)
    words = D_self.index.to_numpy()
    iu = np.triu_indices(len(words), k=1)

    # vectorized SELF (fixed)
    v_self = D_self.values[iu]

    perm_stats = np.zeros(n_perm)

    for i in range(n_perm):
        perm = rng.permutation(len(words))
        Dp = D_other.values[perm][:, perm]
        v_other = Dp[iu]

        mask = np.isfinite(v_self) & np.isfinite(v_other)
        if method == "pearson":
            r, _ = pearsonr(v_self[mask], v_other[mask])
        elif method == "spearman":
            r, _ = spearmanr(v_self[mask], v_other[mask])
        else:
            raise ValueError("method must be 'pearson' or 'spearman'")

        perm_stats[i] = r

    return perm_stats

In [ ]:
perm_p = geometry_permutation_test(D_self, D_other, n_perm=10000, method="pearson")
perm_s = geometry_permutation_test(D_self, D_other, n_perm=10000, method="spearman")

In [ ]:
def permutation_pvalue(r_obs, r_perm):
    return (np.sum(r_perm >= r_obs) + 1) / (len(r_perm) + 1)

pval_p = permutation_pvalue(r_obs_p, perm_p)
pval_s = permutation_pvalue(r_obs_s, perm_s)

print(f"Permutation Pearson p = {pval_p:.4e}")
print(f"Permutation Spearman p = {pval_s:.4e}")

In [ ]:
def p_to_stars(p):
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 0.05: return "*"
    return "n.s."

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

stars = p_to_stars(pval_s)

fig, ax = plt.subplots(figsize=(6, 4))

counts, bins, _ = ax.hist(
    perm_s,
    bins=50,
    color="#3232FF",
    alpha=1,
    edgecolor="none",
    label="Null (permuted)"
)

ax.axvline(
    r_obs_s,
    color="red",
    linewidth=2,
    label="Observed"
)

# ⭐ annotate significance
ymax = counts.max()
ax.text(
    r_obs_s,
    ymax * 0.9,
    f"{stars}\np = {pval_s:.2e}",
    color="red",
    ha="center",
    va="bottom",
    fontsize=12
)

ax.set_xlabel("Spearman geometry correlation", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Permutation test: SELF vs OTHER geometry", fontsize=13, pad=10)

# COSYNE clean
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.1)
ax.spines["bottom"].set_linewidth(1.1)
ax.tick_params(width=1.1, length=4)

ax.legend(frameon=False)

plt.tight_layout()

plt.savefig("perm_test_geometry_spearmanYEU.eps", format="eps", bbox_inches="tight")

plt.show()

# mds

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from scipy.linalg import orthogonal_procrustes

def plot_mds_self_vs_other(
    D_self: pd.DataFrame,
    D_other: pd.DataFrame,
    words,
    n_components=2,
    align=True,
    random_state=0,
    title_suffix=""
):
    """
    Visualize SELF vs OTHER word geometry using MDS.

    D_self, D_other : DataFrames [words x words]
    words           : list of words to include (same order for both)
    n_components    : 2 or 3
    align            : Procrustes-align OTHER to SELF (recommended)
    """

    # subset distance matrices
    D1 = D_self.loc[words, words].values
    D2 = D_other.loc[words, words].values

    # MDS (precomputed distances)
    mds = MDS(
        n_components=n_components,
        dissimilarity="precomputed",
        random_state=random_state,
        normalized_stress="auto",
    )

    X1 = mds.fit_transform(D1)
    X2 = mds.fit_transform(D2)

    # Optional Procrustes alignment (removes rotation/reflection ambiguity)
    if align:
        R, _ = orthogonal_procrustes(X2, X1)
        X2 = X2 @ R

    # ---- plotting ----
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

    for ax, X, label in zip(axes, [X1, X2], ["SELF", "OTHER"]):
        ax.scatter(X[:, 0], X[:, 1], s=50)
        for i, w in enumerate(words):
            ax.text(X[i, 0], X[i, 1], w, fontsize=9, alpha=0.8)
        ax.set_title(f"{label} MDS {title_suffix}".strip())
        ax.axhline(0, color="gray", lw=0.5)
        ax.axvline(0, color="gray", lw=0.5)
        ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()

    return X1, X2

In [ ]:
common_words = sorted(set(D_self.index) & set(D_other.index))
print(len(common_words))
np.random.seed(1)
words = np.random.choice(common_words, size=7, replace=False)

X1, X2 = plot_mds_self_vs_other(
    D_self,
    D_other,
    words=words,
    n_components=2,
    align=True
)

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(X1[:,0], X1[:,1], s=70, color="dodgerblue", label="SELF", alpha=0.8)
plt.scatter(X2[:,0], X2[:,1], s=70, color="crimson", label="OTHER", alpha=0.8)

# optional: draw faint line segments instead of arrows
for i in range(len(words)):
    plt.plot([X1[i,0], X2[i,0]], [X1[i,1], X2[i,1]], color="gray", alpha=0.25, lw=1)

# label only a subset to reduce clutter
for i, w in enumerate(words):
    if i % 3 == 0:   # every third word
        plt.text(X1[i,0], X1[i,1], w, fontsize=9, color="dodgerblue")
        plt.text(X2[i,0], X2[i,1], w, fontsize=9, color="crimson")

plt.axhline(0, color="gray", lw=0.5)
plt.axvline(0, color="gray", lw=0.5)
plt.gca().set_aspect("equal")
plt.legend()
plt.title("SELF vs OTHER MDS")
plt.tight_layout()
plt.show()

In [ ]:
disp = X2 - X1

plt.figure(figsize=(6,6))
plt.scatter(disp[:,0], disp[:,1], s=70, color="gray", alpha=0.8)

for i, w in enumerate(words):
    plt.text(disp[i,0], disp[i,1], w, fontsize=9, alpha=0.8)

plt.axhline(0, color="gray", lw=0.5)
plt.axvline(0, color="gray", lw=0.5)
plt.gca().set_aspect("equal")
plt.title("OTHER - SELF displacement in MDS space")
plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(1,2,1)
plt.imshow(D_self.loc[words, words])
plt.title("SELF RDM")

plt.subplot(1,2,2)
plt.imshow(D_other.loc[words, words])
plt.title("OTHER RDM")

# Network node mama

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import orthogonal_procrustes


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import orthogonal_procrustes


def classical_mds_from_distance(D, n_components=2):
    """
    Classical MDS from a distance matrix.

    Parameters
    ----------
    D : array-like, shape [n, n]
        Symmetric distance matrix.
    n_components : int
        Number of dimensions to return.

    Returns
    -------
    X : ndarray, shape [n, n_components]
        Low-dimensional coordinates.
    """
    D = np.asarray(D, dtype=float)
    n = D.shape[0]

    # Double-centering
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D ** 2) @ J

    # Eigendecomposition
    eigvals, eigvecs = np.linalg.eigh(B)

    # Sort descending
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Keep only positive eigenvalues
    pos = eigvals > 0
    eigvals = eigvals[pos][:n_components]
    eigvecs = eigvecs[:, pos][:, :n_components]

    X = eigvecs * np.sqrt(eigvals)
    return X


def draw_network_from_coords(
    ax,
    X,
    words,
    D,
    title="",
    edge_mode="all",
    k=2,
    distance_percentile=25,
    node_size=600,
    node_color="red",
    node_alpha=0.5,
    edge_color="gray",
    edge_alpha=0.35,
    edge_lw=1.0,
    font_size=10,
    label_alpha=0.9,
):
    """
    Draw a network-like visualization from 2D coords and distance matrix.

    Parameters
    ----------
    edge_mode : {"all", "knn", "threshold"}
        "all"       : connect every node to every other node
        "knn"       : connect each node to k nearest neighbors
        "threshold" : connect pairs below a distance percentile
    """
    X = np.asarray(X, dtype=float)
    D = np.asarray(D, dtype=float)
    n = len(words)

    # ----- draw edges -----
    drawn = set()

    if edge_mode == "all":
        for i in range(n):
            for j in range(i + 1, n):
                ax.plot(
                    [X[i, 0], X[j, 0]],
                    [X[i, 1], X[j, 1]],
                    lw=edge_lw,
                    color=edge_color,
                    alpha=edge_alpha,
                    zorder=1,
                )

    elif edge_mode == "knn":
        for i in range(n):
            nbrs = np.argsort(D[i])[1 : k + 1]  # skip self
            for j in nbrs:
                a, b = sorted((i, j))
                if (a, b) not in drawn:
                    drawn.add((a, b))
                    ax.plot(
                        [X[a, 0], X[b, 0]],
                        [X[a, 1], X[b, 1]],
                        lw=edge_lw,
                        color=edge_color,
                        alpha=edge_alpha,
                        zorder=1,
                    )

    elif edge_mode == "threshold":
        vals = D[np.triu_indices(n, k=1)]
        thresh = np.percentile(vals, distance_percentile)
        for i in range(n):
            for j in range(i + 1, n):
                if D[i, j] <= thresh:
                    ax.plot(
                        [X[i, 0], X[j, 0]],
                        [X[i, 1], X[j, 1]],
                        lw=edge_lw,
                        color=edge_color,
                        alpha=edge_alpha,
                        zorder=1,
                    )
    else:
        raise ValueError("edge_mode must be 'all', 'knn', or 'threshold'")

    # ----- draw nodes -----
    ax.scatter(
        X[:, 0],
        X[:, 1],
        s=node_size,
        color=node_color,
        alpha=node_alpha,
        zorder=2,
    )

    # ----- labels -----
    for i, w in enumerate(words):
        ax.text(
            X[i, 0],
            X[i, 1],
            str(w),
            fontsize=font_size,
            alpha=label_alpha,
            ha="center",
            va="center",
            zorder=3,
        )

    ax.set_title(title)
    ax.set_aspect("equal")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])


def plot_mds_network_self_vs_other(
    D_self: pd.DataFrame,
    D_other: pd.DataFrame,
    words,
    align=True,
    edge_mode="knn",
    k=2,
    distance_percentile=25,
    title_suffix="",
    figsize=(6, 6),
    save_prefix=None,
    show=True,
    node_size=600,
    node_alpha=0.5,
    edge_alpha=0.7,
    edge_lw=1.5,
    font_size=10,
):
    """
    Network-style classical MDS plot for SELF vs OTHER as two separate figures.

    Parameters
    ----------
    D_self, D_other : pd.DataFrame
        Square distance matrices with word labels as index/columns.
    words : list-like
        Words to include, in the same order for both matrices.
    align : bool
        If True, Procrustes-align OTHER to SELF.
    edge_mode : {"knn", "threshold"}
        How to draw network edges.
    k : int
        Number of nearest neighbors if edge_mode="knn".
    distance_percentile : float
        Percentile threshold if edge_mode="threshold".
    title_suffix : str
        Text appended to plot titles.
    figsize : tuple
        Figure size for each separate figure.
    save_prefix : str or None
        If provided, saves:
        - {save_prefix}_SELF.eps
        - {save_prefix}_OTHER.eps
    show : bool
        If True, display figures.
    node_size : float
        Node size in scatter plot.
    node_alpha : float
        Node transparency.
    edge_alpha : float
        Edge transparency.
    edge_lw : float
        Edge line width.
    font_size : float
        Word label font size.

    Returns
    -------
    X1, X2 : ndarray
        2D coordinates for SELF and OTHER.
    """
    words = list(words)

    # subset and enforce same order
    D1 = D_self.loc[words, words].values
    D2 = D_other.loc[words, words].values

    # classical MDS
    X1 = classical_mds_from_distance(D1, n_components=2)
    X2 = classical_mds_from_distance(D2, n_components=2)

    # optional Procrustes alignment
    if align:
        R, _ = orthogonal_procrustes(X2, X1)
        X2 = X2 @ R

    # ----- shared axis limits -----
    xmin = min(X1[:, 0].min(), X2[:, 0].min())
    xmax = max(X1[:, 0].max(), X2[:, 0].max())
    ymin = min(X1[:, 1].min(), X2[:, 1].min())
    ymax = max(X1[:, 1].max(), X2[:, 1].max())

    # padding so nodes/labels aren't clipped
    xr = xmax - xmin
    yr = ymax - ymin
    pad_x = 0.10 * xr if xr > 0 else 0.5
    pad_y = 0.10 * yr if yr > 0 else 0.5

    xmin -= pad_x
    xmax += pad_x
    ymin -= pad_y
    ymax += pad_y

    # ----- SELF figure -----
    fig1, ax1 = plt.subplots(figsize=figsize)
    draw_network_from_coords(
        ax=ax1,
        X=X1,
        words=words,
        D=D1,
        title=f"SELF {title_suffix}".strip(),
        edge_mode=edge_mode,
        k=k,
        distance_percentile=distance_percentile,
        node_size=node_size,
        node_color="red",
        node_alpha=node_alpha,
        edge_color="gray",
        edge_alpha=edge_alpha,
        edge_lw=edge_lw,
        font_size=font_size,
    )
    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(ymin, ymax)
    plt.tight_layout()

    if save_prefix is not None:
        fig1.savefig(f"{save_prefix}_SELF.eps", format="eps")

    if show:
        plt.show()
    else:
        plt.close(fig1)

    # ----- OTHER figure -----
    fig2, ax2 = plt.subplots(figsize=figsize)
    draw_network_from_coords(
        ax=ax2,
        X=X2,
        words=words,
        D=D2,
        title=f"OTHER {title_suffix}".strip(),
        edge_mode=edge_mode,
        k=k,
        distance_percentile=distance_percentile,
        node_size=node_size,
        node_color="blue",
        node_alpha=node_alpha,
        edge_color="gray",
        edge_alpha=edge_alpha,
        edge_lw=edge_lw,
        font_size=font_size,
    )
    ax2.set_xlim(xmin, xmax)
    ax2.set_ylim(ymin, ymax)
    plt.tight_layout()

    if save_prefix is not None:
        fig2.savefig(f"{save_prefix}_OTHER.eps", format="eps")

    if show:
        plt.show()
    else:
        plt.close(fig2)

    return X1, X2



In [ ]:
common_words = sorted(set(D_self.index) & set(D_other.index))
# np.random.seed(0)
rng = np.random.default_rng()   # new sample each run
words = rng.choice(common_words, size=5, replace=False)

X1, X2 = plot_mds_network_self_vs_other(
    D_self,
    D_other,
    words=words,
    align=True,
    edge_mode="all",   # or "threshold"
    title_suffix="Matched words"
)

In [ ]:
patient_id = "PTYFF_4"

common_words = sorted(set(D_self.index) & set(D_other.index))
# rng = np.random.default_rng(0)   # fixed sample if you want reproducible figures
words = rng.choice(common_words, size=6, replace=False)

X1, X2 = plot_mds_network_self_vs_other(
    D_self,
    D_other,
    words=words,
    align=True,
    edge_mode="all",
    font_size=20,
    title_suffix="Matched words",
    node_size=1500,
    node_alpha=0.4,
    save_prefix=f"{patient_id}_mds",
    
)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import to_rgb
from scipy.linalg import orthogonal_procrustes

# ── recompute X1 / X2 from the distance matrices so this cell is self-contained ──
_words_movie = list(words)          # the word subset chosen in cell-23
D1 = D_self.loc[_words_movie, _words_movie].values
D2 = D_other.loc[_words_movie, _words_movie].values

def _cmds(D, n=2):
    D = np.asarray(D, float)
    n_pts = D.shape[0]
    J = np.eye(n_pts) - np.ones((n_pts, n_pts)) / n_pts
    B = -0.5 * J @ (D ** 2) @ J
    vals, vecs = np.linalg.eigh(B)
    idx = np.argsort(vals)[::-1]
    vals, vecs = vals[idx], vecs[:, idx]
    pos = vals > 0
    return vecs[:, pos][:, :n] * np.sqrt(vals[pos][:n])

X1 = _cmds(D1)
X2 = _cmds(D2)
R, _ = orthogonal_procrustes(X2, X1)
X2 = X2 @ R

# ── animation parameters ──
N_FRAMES    = 80
HOLD_FRAMES = 15
FPS         = 30
SAVE_PATH   = f"{patient_id}_rotation.mp4"

t_fwd        = np.linspace(0, 1, N_FRAMES)
t_hold_end   = np.ones(HOLD_FRAMES)
t_back       = np.linspace(1, 0, N_FRAMES)
t_hold_start = np.zeros(HOLD_FRAMES)
t_seq = np.concatenate([t_hold_start, t_fwd, t_hold_end, t_back])

all_pts = np.vstack([X1, X2])
pad  = 0.15 * all_pts.ptp(axis=0).max()
xlim = (all_pts[:, 0].min() - pad, all_pts[:, 0].max() + pad)
ylim = (all_pts[:, 1].min() - pad, all_pts[:, 1].max() + pad)

# ── colours ──
COLOR_SELF  = np.array(to_rgb("red"))   # (0.545, 0.0, 0.0)
COLOR_OTHER = np.array(to_rgb("blue"))      # (0.0, 0.0, 1.0)
NODE_ALPHA  = 0.4

# ── set up figure ──
fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal")
ax.axis("off")

title_txt = ax.set_title("", fontsize=14, pad=10)

n_words    = len(_words_movie)
edge_pairs = [(i, j) for i in range(n_words) for j in range(i + 1, n_words)]

edge_lines = []
for _ in edge_pairs:
    ln, = ax.plot([], [], color="gray", alpha=0.35, lw=1.2, zorder=1)
    edge_lines.append(ln)

scat  = ax.scatter([], [], s=1500, zorder=2)
texts = [ax.text(0, 0, w, ha="center", va="center",
                 fontsize=20, fontweight="bold", zorder=3)
         for w in _words_movie]

def update(frame_idx):
    t = t_seq[frame_idx]
    X = (1 - t) * X1 + t * X2

    c    = (1 - t) * COLOR_SELF + t * COLOR_OTHER
    rgba = np.column_stack([np.tile(c, (n_words, 1)),
                            np.full(n_words, NODE_ALPHA)])
    scat.set_offsets(X)
    scat.set_facecolor(rgba)

    for ln, (i, j) in zip(edge_lines, edge_pairs):
        ln.set_data([X[i, 0], X[j, 0]], [X[i, 1], X[j, 1]])

    for k, txt in enumerate(texts):
        txt.set_position((X[k, 0], X[k, 1]))

    if t < 0.05:
        lbl = "SELF"
    elif t > 0.95:
        lbl = "OTHER"
    else:
        lbl = f"SELF → OTHER  ({t:.0%})"
    title_txt.set_text(lbl)

    return [scat, title_txt] + edge_lines + texts

ani = animation.FuncAnimation(
    fig, update,
    frames=len(t_seq),
    interval=1000 / FPS,
    blit=True,
)

writer = animation.FFMpegWriter(fps=FPS, bitrate=2000)
ani.save(SAVE_PATH, writer=writer, dpi=150)
plt.close(fig)
print(f"Saved: {SAVE_PATH}")
